In [1]:
import pandas as pd
import psycopg2

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
import psycopg2

connection = psycopg2.connect(
    host="localhost",
    port="5432",
    database="ai_helpdesk",
    user="postgres",
    password="Bindu@1412"
)

print("Connected to PostgreSQL successfully!")

Connected to PostgreSQL successfully!


In [3]:
df = pd.read_csv("../data/raw/tickets.csv")

print("Ticket dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Ticket dataset loaded successfully!
Rows: 500
Columns: 12


,ticket_id,employee_name,department,employee_type,subject,description,category,priority,status,created_at,resolved_at,resolution_time_hours
0,1,Employee_37,Sales,Permanent,Internet connection is very slow,Internet connection is very slow,Network,High,Resolved,2026-09-05 06:13:51,2026-09-05 17:06:39,10.88
1,2,Employee_47,Finance,Intern,Software is showing an error,Software is showing an error,Software,Medium,Open,2026-04-25 12:29:51,NaN,27.75
2,3,Employee_48,HR,Intern,Mouse is not responding,Mouse is not responding,Hardware,Low,Resolved,2026-07-03 08:02:51,2026-07-04 06:50:51,22.80
3,4,Employee_41,Finance,Contract,Application is crashing,Application is crashing,Software,Medium,In Progress,2026-05-09 20:05:51,NaN,22.59
4,5,Employee_86,Finance,Intern,Unable to send email,Unable to send email,Email,High,Resolved,2026-08-23 19:18:51,2026-08-23 20:29:03,1.17


In [4]:
print("Dataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nFirst 5 rows:")
df.head()

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ticket_id              500 non-null    int64  
 1   employee_name          500 non-null    object 
 2   department             500 non-null    object 
 3   employee_type          500 non-null    object 
 4   subject                500 non-null    object 
 5   description            500 non-null    object 
 6   category               500 non-null    object 
 7   priority               500 non-null    object 
 8   status                 500 non-null    object 
 9   created_at             500 non-null    object 
 10  resolved_at            173 non-null    object 
 11  resolution_time_hours  500 non-null    float64
dtypes: float64(1), int64(1), object(10)
memory usage: 47.0+ KB

Missing values:
ticket_id                  0
employee_name              0
departm

,ticket_id,employee_name,department,employee_type,subject,description,category,priority,status,created_at,resolved_at,resolution_time_hours
0,1,Employee_37,Sales,Permanent,Internet connection is very slow,Internet connection is very slow,Network,High,Resolved,2026-09-05 06:13:51,2026-09-05 17:06:39,10.88
1,2,Employee_47,Finance,Intern,Software is showing an error,Software is showing an error,Software,Medium,Open,2026-04-25 12:29:51,NaN,27.75
2,3,Employee_48,HR,Intern,Mouse is not responding,Mouse is not responding,Hardware,Low,Resolved,2026-07-03 08:02:51,2026-07-04 06:50:51,22.80
3,4,Employee_41,Finance,Contract,Application is crashing,Application is crashing,Software,Medium,In Progress,2026-05-09 20:05:51,NaN,22.59
4,5,Employee_86,Finance,Intern,Unable to send email,Unable to send email,Email,High,Resolved,2026-08-23 19:18:51,2026-08-23 20:29:03,1.17


In [5]:
# Remove duplicate rows
df = df.drop_duplicates().copy()

# Fill missing text values
df["employee_name"] = df["employee_name"].fillna("Unknown")
df["department"] = df["department"].fillna("Unknown")
df["employee_type"] = df["employee_type"].fillna("Unknown")
df["subject"] = df["subject"].fillna("")
df["description"] = df["description"].fillna("")

# Convert date columns
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df["resolved_at"] = pd.to_datetime(df["resolved_at"], errors="coerce")

print("Data cleaning completed!")
print("Rows after cleaning:", len(df))
print("\nMissing values:")
print(df.isnull().sum())

Data cleaning completed!
Rows after cleaning: 500

Missing values:
ticket_id                  0
employee_name              0
department                 0
employee_type              0
subject                    0
description                0
category                   0
priority                   0
status                     0
created_at                 0
resolved_at              327
resolution_time_hours      0
dtype: int64


In [6]:
employee_columns = [
    "employee_name",
    "department",
    "employee_type"
]

employees_df = df[employee_columns].drop_duplicates().reset_index(drop=True)

print("Employee data prepared successfully!")
print("Number of unique employees:", len(employees_df))

employees_df.head()

Employee data prepared successfully!
Number of unique employees: 421


,employee_name,department,employee_type
0,Employee_37,Sales,Permanent
1,Employee_47,Finance,Intern
2,Employee_48,HR,Intern
3,Employee_41,Finance,Contract
4,Employee_86,Finance,Intern


In [7]:
ticket_columns = [
    "ticket_id",
    "employee_name",
    "subject",
    "description",
    "category",
    "priority",
    "status",
    "created_at",
    "resolved_at",
    "resolution_time_hours"
]

tickets_df = df[ticket_columns].copy()

print("Ticket data prepared successfully!")
print("Number of tickets:", len(tickets_df))

tickets_df.head()

Ticket data prepared successfully!
Number of tickets: 500


,ticket_id,employee_name,subject,description,category,priority,status,created_at,resolved_at,resolution_time_hours
0,1,Employee_37,Internet connection is very slow,Internet connection is very slow,Network,High,Resolved,2026-09-05 06:13:51,2026-09-05 17:06:39,10.88
1,2,Employee_47,Software is showing an error,Software is showing an error,Software,Medium,Open,2026-04-25 12:29:51,NaT,27.75
2,3,Employee_48,Mouse is not responding,Mouse is not responding,Hardware,Low,Resolved,2026-07-03 08:02:51,2026-07-04 06:50:51,22.80
3,4,Employee_41,Application is crashing,Application is crashing,Software,Medium,In Progress,2026-05-09 20:05:51,NaT,22.59
4,5,Employee_86,Unable to send email,Unable to send email,Email,High,Resolved,2026-08-23 19:18:51,2026-08-23 20:29:03,1.17


In [10]:
# Reset the failed PostgreSQL transaction
connection.rollback()

cursor = connection.cursor()

cursor.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'employees'
    ORDER BY ordinal_position;
""")

columns = cursor.fetchall()

print("Employees table columns:")

for column in columns:
    print(column)

Employees table columns:
('employee_id', 'integer')
('name', 'character varying')
('department', 'character varying')
('employee_type', 'character varying')


In [12]:
connection.rollback()

cursor = connection.cursor()

cursor.execute("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = 'employees'
    ORDER BY ordinal_position;
""")

columns = cursor.fetchall()

print("Exact employees table columns:")

for column in columns:
    print(repr(column[0]))

Exact employees table columns:
'employee_id'
'name'
'department'
'employee_type'


In [13]:
connection.rollback()

cursor = connection.cursor()

employee_insert_query = """
INSERT INTO employees (name, department, employee_type)
VALUES (%s, %s, %s)
"""

for _, row in employees_df.iterrows():
    cursor.execute(
        employee_insert_query,
        (
            row["employee_name"],
            row["department"],
            row["employee_type"]
        )
    )

connection.commit()

print("Employee data inserted into PostgreSQL successfully!")

Employee data inserted into PostgreSQL successfully!


In [16]:
connection.rollback()

cursor = connection.cursor()

cursor.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = 'tickets'
    ORDER BY ordinal_position;
""")

columns = cursor.fetchall()

print("Exact tickets table columns and data types:")

for column in columns:
    print(column)

Exact tickets table columns and data types:
('ticket_id', 'integer')
('employee_id', 'integer')
('subject', 'character varying')
('description', 'text')
('category', 'character varying')
('priority', 'character varying')
('status', 'character varying')
('created_at', 'timestamp without time zone')
('resolved_at', 'timestamp without time zone')
('resolution_time_hours', 'numeric')


In [17]:
connection.rollback()

# Convert date columns to proper datetime values
tickets_df["created_at"] = pd.to_datetime(
    tickets_df["created_at"],
    errors="coerce"
)

tickets_df["resolved_at"] = pd.to_datetime(
    tickets_df["resolved_at"],
    errors="coerce"
)

cursor = connection.cursor()

ticket_insert_query = """
INSERT INTO tickets (
    ticket_id,
    employee_id,
    subject,
    description,
    category,
    priority,
    status,
    created_at,
    resolved_at,
    resolution_time_hours
)
SELECT
    %s,
    employee_id,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s,
    %s
FROM employees
WHERE name = %s
LIMIT 1
"""

for _, row in tickets_df.iterrows():

    created_at = row["created_at"].to_pydatetime() \
        if pd.notna(row["created_at"]) else None

    resolved_at = row["resolved_at"].to_pydatetime() \
        if pd.notna(row["resolved_at"]) else None

    cursor.execute(
        ticket_insert_query,
        (
            row["ticket_id"],
            row["subject"],
            row["description"],
            row["category"],
            row["priority"],
            row["status"],
            created_at,
            resolved_at,
            row["resolution_time_hours"],
            row["employee_name"]
        )
    )

connection.commit()

print("Ticket data inserted into PostgreSQL successfully!")

Ticket data inserted into PostgreSQL successfully!


In [18]:
connection.rollback()

cursor = connection.cursor()

cursor.execute("""
    SELECT COUNT(*)
    FROM tickets;
""")

ticket_count = cursor.fetchone()[0]

print("Total tickets in PostgreSQL:", ticket_count)

cursor.execute("""
    SELECT
        ticket_id,
        employee_id,
        subject,
        category,
        priority,
        status
    FROM tickets
    ORDER BY ticket_id
    LIMIT 5;
""")

rows = cursor.fetchall()

print("\nFirst 5 tickets:")
for row in rows:
    print(row)

Total tickets in PostgreSQL: 500

First 5 tickets:
(1, 1, 'Internet connection is very slow', 'Network', 'High', 'Resolved')
(2, 2, 'Software is showing an error', 'Software', 'Medium', 'Open')
(3, 3, 'Mouse is not responding', 'Hardware', 'Low', 'Resolved')
(4, 4, 'Application is crashing', 'Software', 'Medium', 'In Progress')
(5, 5, 'Unable to send email', 'Email', 'High', 'Resolved')


In [19]:
connection.rollback()

cursor = connection.cursor()

cursor.execute("""
    SELECT category, COUNT(*) AS ticket_count
    FROM tickets
    GROUP BY category
    ORDER BY ticket_count DESC;
""")

category_results = cursor.fetchall()

print("Tickets by category:")

for row in category_results:
    print(f"{row[0]}: {row[1]}")

Tickets by category:
Account: 93
Network: 92
Email: 81
Software: 79
Security: 79
Hardware: 76


In [20]:
def run_etl_pipeline():
    print("Starting ETL pipeline...")

    # Extract
    data = pd.read_csv("../data/raw/tickets.csv")
    print("1. Data extracted:", len(data), "rows")

    # Transform
    data = data.drop_duplicates().copy()

    data["created_at"] = pd.to_datetime(
        data["created_at"],
        errors="coerce"
    )

    data["resolved_at"] = pd.to_datetime(
        data["resolved_at"],
        errors="coerce"
    )

    data["employee_name"] = data["employee_name"].fillna("Unknown")
    data["department"] = data["department"].fillna("Unknown")
    data["employee_type"] = data["employee_type"].fillna("Unknown")

    print("2. Data transformed:", len(data), "rows")

    # Load
    print("3. Data ready for PostgreSQL loading")

    return data


etl_data = run_etl_pipeline()

print("\nETL pipeline preparation completed!")

Starting ETL pipeline...
1. Data extracted: 500 rows
2. Data transformed: 500 rows
3. Data ready for PostgreSQL loading

ETL pipeline preparation completed!


In [21]:
print("Running complete ETL pipeline...")

etl_data = run_etl_pipeline()

print("\nETL pipeline executed successfully!")
print("Final rows ready for database:", len(etl_data))

Running complete ETL pipeline...
Starting ETL pipeline...
1. Data extracted: 500 rows
2. Data transformed: 500 rows
3. Data ready for PostgreSQL loading

ETL pipeline executed successfully!
Final rows ready for database: 500


In [22]:
connection.rollback()

cursor = connection.cursor()

# Count employees
cursor.execute("SELECT COUNT(*) FROM employees;")
employee_count = cursor.fetchone()[0]

# Count tickets
cursor.execute("SELECT COUNT(*) FROM tickets;")
ticket_count = cursor.fetchone()[0]

# Check ticket categories
cursor.execute("""
    SELECT category, COUNT(*)
    FROM tickets
    GROUP BY category
    ORDER BY category;
""")

category_results = cursor.fetchall()

print("===== FINAL DATABASE VERIFICATION =====")
print("Total employees:", employee_count)
print("Total tickets:", ticket_count)

print("\nTickets by category:")
for category, count in category_results:
    print(f"{category}: {count}")

print("\nPhase 4 completed successfully!")

===== FINAL DATABASE VERIFICATION =====
Total employees: 421
Total tickets: 500

Tickets by category:
Account: 93
Email: 81
Hardware: 76
Network: 92
Security: 79
Software: 79

Phase 4 completed successfully!
